<a href="https://colab.research.google.com/github/zahid1704/ML_Project/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zahid1704/ML_Project/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

one row in fact_content_daily_performance is one page (content_hash_id) for one client (client_hash_id) on one day (report_date). I will collapse March into one row per page per client, so each row summarizes that page’s signals over the month. March 2026 (month=2026-03), from 2026-03-01 to 2026-03-31. I chose a mid-panel month because the final month (June 2026) is the sealed test month, and the daily data starts in January 2025, so March has history behind it for most clients.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
MONTH = "2026-03"
START_DATE, END_DATE = "2026-03-01", "2026-03-31"
RAW_GRAIN = ["report_date", "client_hash_id", "content_hash_id"]
ANALYSIS_GRAIN = ["client_hash_id", "content_hash_id"]

print("Month:", MONTH, "|", START_DATE, "to", END_DATE)
print("Raw grain:", RAW_GRAIN)
print("Analysis grain (one row per page per client):", ANALYSIS_GRAIN)


Month: 2026-03 | 2026-03-01 to 2026-03-31
Raw grain: ['report_date', 'client_hash_id', 'content_hash_id']
Analysis grain (one row per page per client): ['client_hash_id', 'content_hash_id']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label (proxy): low_ctr_for_position, which is 1 when a page’s monthly CTR is below the median CTR of pages in the same position tier. CTR is computed as total gsc_clicks divided by total gsc_impressions for the page over the month. This is a proxy computed from observed values, not a future outcome.

Features (five): gsc_impressions (demand), gsc_avg_position (rank), word_count and content_age_days (from dim_content, to confirm), and engagement rate (ga4_engaged_sessions / ga4_sessions, available only where analytics tracking exists).

Context (not model inputs): client_hash_id, content_hash_id, report_date, gsc_data_available, ga4_data_available, client_has_gsc, client_has_ga4.

Excluded, with reasons:

gsc_clicks and computed CTR: the label is built from them, so using either as a feature would leak the answer.
sessions_ai and the per-platform AI columns (ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other): AI-referral rows are very sparse, so they would be mostly empty and misleading.
Raw URLs, queries, titles, client names: removed from the release for privacy, and I will not try to reconstruct them.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fields = {
    "label": ["low_ctr_for_position  (gsc_clicks / gsc_impressions vs. median within position tier)"],
    "feature": ["gsc_impressions", "gsc_avg_position", "word_count (confirm in dim_content)",
                "content_age_days (confirm in dim_content)", "engagement_rate = ga4_engaged_sessions / ga4_sessions"],
    "context": ["client_hash_id", "content_hash_id", "report_date", "gsc_data_available",
                "ga4_data_available", "client_has_gsc", "client_has_ga4"],
    "excluded": {
        "gsc_clicks": "label is built from it, would leak",
        "ctr (gsc_clicks / gsc_impressions)": "this is the label's basis, would leak",
        "sessions_ai + ai_* columns": "too sparse to be reliable",
    },
}
for bucket, cols in fields.items():
    print(bucket, "->", cols)

label -> ['low_ctr_for_position  (gsc_clicks / gsc_impressions vs. median within position tier)']
feature -> ['gsc_impressions', 'gsc_avg_position', 'word_count (confirm in dim_content)', 'content_age_days (confirm in dim_content)', 'engagement_rate = ga4_engaged_sessions / ga4_sessions']
context -> ['client_hash_id', 'content_hash_id', 'report_date', 'gsc_data_available', 'ga4_data_available', 'client_has_gsc', 'client_has_ga4']
excluded -> {'gsc_clicks': 'label is built from it, would leak', 'ctr (gsc_clicks / gsc_impressions)': "this is the label's basis, would leak", 'sessions_ai + ai_* columns': 'too sparse to be reliable'}


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

con = duckdb.connect()
token = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected")

Connected


In [33]:
files = con.sql(f"SELECT * FROM glob('{REL}/fact_content_daily_performance/month=2026-03/*')").df()
print(files)

                                                file
0  hf://datasets/FlyRank/internship-warehouse/fac...


In [34]:
con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""")
print(con.sql("DESCRIBE march").df()[["column_name", "column_type"]])

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [35]:
con.execute(f"CREATE OR REPLACE VIEW content AS SELECT * FROM read_parquet('{REL}/dim_content.parquet')")
print(con.sql("DESCRIBE content").df()[["column_name", "column_type"]])

                   column_name column_type
0               client_hash_id     VARCHAR
1              content_hash_id     VARCHAR
2              keyword_hash_id     VARCHAR
3                  url_hash_id     VARCHAR
4           keyword_char_count      BIGINT
5          keyword_token_count      BIGINT
6               url_char_count      BIGINT
7         content_created_date        DATE
8         content_updated_date        DATE
9                 content_type     VARCHAR
10               search_volume      BIGINT
11                 competition      DOUBLE
12           competition_level     VARCHAR
13                         cpc      DOUBLE
14                 main_intent     VARCHAR
15                   backlinks      BIGINT
16              category_count      BIGINT
17        keyword_created_date        DATE
18               provider_used     VARCHAR
19                  model_used     VARCHAR
20                  char_count      BIGINT
21                  word_count      BIGINT
22         

In [36]:
con.sql("SELECT * FROM march LIMIT 5").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [37]:
con.sql("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys
FROM march
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_keys
0,9841378,9841378


In [38]:
con.sql("""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
       COUNT(DISTINCT client_hash_id) AS clients, COUNT(DISTINCT content_hash_id) AS pages
FROM march
""").df()

,n_rows,first_day,last_day,clients,pages
0,9841378,2026-03-01,2026-03-31,55,331437


In [39]:
con.sql("""
SELECT COUNT(*) AS all_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4,
       ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 2) AS pct_ga4,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc
FROM march
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,rows_with_ga4,pct_ga4,rows_with_gsc
0,9841378,413966,4.21,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell me:

Cause. I only observe what happened. I can’t say that changing a page’s length or age causes more clicks, because no experiment was run.
Unbalanced history. Different clients have different amounts of history, because each client’s tracking began at a different time. A single month like March 2026 includes clients with long histories and clients that just started, so patterns may reflect client maturity and not page quality.
Search-only early rows. Rows from before a client’s analytics tracking started have ga4_data_available = FALSE and contain search data only. My engagement feature exists only where analytics is available, so it covers a subset of rows (Query 3 shows the share), and missing engagement means “not tracked,” not “no engagement.”
Window overlaps. My label and features all come from the same month. This means I can describe how signals go together within March, but I can’t claim that a signal predicts later decline. A future-window label would need separate, non-overlapping feature and target windows.
One month only. March may be affected by seasonality, so I can’t say whether the patterns hold in other months.

Named limitation of my slice: only 4.21% of March rows have analytics data (from Query 3), so engagement results describe that subset, not all pages. Only about 36.7% of March rows contain search data, so my label and features describe pages that appeared in search, not every page in the inventory.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.